## DLinear — Walmart Store Sales Forecasting

**Before Save & Run All on Kaggle:**
1. Attach competition input: *Walmart Recruiting - Store Sales Forecasting*
2. Settings → **Accelerator: GPU T4 x2** (optional; CPU works)
3. Settings → **Internet: On** (pip + DagsHub)
4. Add Kaggle secret `WANDB_API_KEY` — cell **#12** logs full training curves to W&B on **Run All**

**Quick run (~30–45 min, leave for 1h):** cell **#4** → `LONG_RUN = False` (top 100 series).

**Overnight run:** cell **#4** → `LONG_RUN = True` (~1–3 hours, all series).

**Reliability while you are away:**
- Progress → `/kaggle/working/dlinear_progress.log`
- Checkpoints → `/kaggle/working/checkpoints/`
- Success marker → `/kaggle/working/RUN_COMPLETE.txt`

In [ ]:
#1
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
#2
NOTEBOOK_VERSION = 'dlinear_v1_long_run'
!pip install dagshub mlflow wandb -q
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118

import json, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mlflow
import dagshub

warnings.filterwarnings('ignore')

WORK_DIR = Path('/kaggle/working')
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = WORK_DIR / 'dlinear_progress.log'
RUN_COMPLETE = WORK_DIR / 'RUN_COMPLETE.txt'

if RUN_COMPLETE.exists():
    RUN_COMPLETE.unlink()

def log_progress(msg: str):
    line = f"[{datetime.utcnow().isoformat()}Z] {msg}"
    print(line)
    with open(PROGRESS_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

def pick_device():
    if not torch.cuda.is_available():
        log_progress('CUDA not available -> CPU')
        return torch.device('cpu'), False
    try:
        x = torch.randn(8, 52, device='cuda')
        layer = nn.Linear(52, 8).cuda()
        _ = layer(x)
        torch.cuda.synchronize()
        log_progress(
            f"CUDA OK: {torch.cuda.get_device_name(0)} | "
            f"torch={torch.__version__} | cuda={torch.version.cuda}"
        )
        return torch.device('cuda'), True
    except Exception as e:
        log_progress(f'CUDA smoke test failed ({e}) -> CPU fallback')
        return torch.device('cpu'), False

device, USE_CUDA = pick_device()
if USE_CUDA:
    torch.cuda.manual_seed_all(SEED)

_secrets = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None

_dagshub_token = None
if _secrets is not None:
    try:
        _dagshub_token = _secrets.get_secret('DAGSHUB_USER_TOKEN')
    except Exception:
        _dagshub_token = None

if _dagshub_token:
    dagshub.init(
        repo_owner='lshek22',
        repo_name='walmart-recruiting-store-sales-forecasting',
        mlflow=True,
        token=_dagshub_token,
    )
    log_progress('DagsHub auth: token')
else:
    dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)
    log_progress('DagsHub auth: oauth')

mlflow.set_experiment('DLinear_Training')

USE_WANDB = False
try:
    import wandb
    wandb.login(key=_secrets.get_secret('WANDB_API_KEY'))
    USE_WANDB = True
    log_progress('W&B enabled')
except Exception as e:
    log_progress(f'W&B disabled: {e}')

log_progress(f'Notebook version: {NOTEBOOK_VERSION}')
log_progress('Setup complete')

In [ ]:
#3
DATA_DIR = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting'

train = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')
features = pd.read_csv(f'{DATA_DIR}/features.csv.zip')

for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])

train = train.merge(stores, on='Store', how='left')
train = train.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')
test = test.merge(stores, on='Store', how='left')
test = test.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
train[md_cols] = train[md_cols].fillna(0)
test[md_cols] = test[md_cols].fillna(0)

for col in ['CPI', 'Unemployment']:
    train[col] = train.groupby('Store')[col].transform(lambda x: x.ffill().bfill())
    test[col] = test.groupby('Store')[col].transform(lambda x: x.ffill().bfill())

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)
train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

log_progress(f'Train shape: {train.shape} | Test shape: {test.shape}')
log_progress(f'Unique series: {train.groupby(["Store","Dept"]).ngroups}')

In [ ]:
#4
LONG_RUN = False  # False = ~30-45 min (safe for 1h away) | True = ~1-3h all series

SPLIT_DATE = pd.Timestamp('2012-07-01')
SEQ_LEN = 52
PRED_LEN = 8
MAX_SERIES = None if LONG_RUN else 100

train_df = train[train['Date'] < SPLIT_DATE].copy()
val_df = train[train['Date'] >= SPLIT_DATE].copy()

def holiday_weight(series):
    return np.where(series.fillna(0).astype(bool), 5, 1).astype(np.float32)

train_df['holiday_weight'] = holiday_weight(train_df['IsHoliday'])
val_df['holiday_weight'] = holiday_weight(val_df['IsHoliday'])

log_progress(f'LONG_RUN={LONG_RUN} | MAX_SERIES={"all" if MAX_SERIES is None else MAX_SERIES}')
log_progress(
    f'Train: {train_df.Date.min().date()} -> {train_df.Date.max().date()} | '
    f'Val: {val_df.Date.min().date()} -> {val_df.Date.max().date()}'
)

with mlflow.start_run(run_name='DLinear_Cleaning'):
    mlflow.log_params({
        'split_date': str(SPLIT_DATE.date()),
        'seq_len': SEQ_LEN,
        'pred_len': PRED_LEN,
        'max_series': MAX_SERIES if MAX_SERIES is not None else 'all',
        'long_run': LONG_RUN,
        'target_scaling': 'per_series_mean_std',
        'model_type': 'DLinear decomposition-linear',
    })
log_progress('DLinear_Cleaning logged')

In [ ]:
#5
class MovingAvg(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.kernel_size = kernel_size

    def forward(self, x):
        # x: [B, L, 1]
        pad = (self.kernel_size - 1) // 2
        front = x[:, 0:1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, pad, 1)
        x_pad = torch.cat([front, x, end], dim=1)
        x_pad = x_pad.permute(0, 2, 1)
        out = nn.functional.avg_pool1d(x_pad, kernel_size=self.kernel_size, stride=1)
        return out.permute(0, 2, 1)


class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend


class DLinear(nn.Module):
    """DLinear: decomposition + linear forecast (Zeng et al., 2023)."""

    def __init__(self, seq_len: int, pred_len: int, kernel_size: int = 25):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        ks = min(kernel_size, seq_len if seq_len % 2 == 1 else seq_len - 1)
        if ks < 3:
            ks = 3
        if ks % 2 == 0:
            ks -= 1
        self.kernel_size = ks
        self.decomposition = SeriesDecomp(self.kernel_size)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.linear_trend = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        # x: [B, seq_len]
        x3 = x.unsqueeze(-1)
        seasonal, trend = self.decomposition(x3)
        seasonal = seasonal.squeeze(-1)
        trend = trend.squeeze(-1)
        return self.linear_seasonal(seasonal) + self.linear_trend(trend)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

log_progress('DLinear model classes ready')

In [ ]:
#6
class WalmartDLinearDataset(Dataset):
    def __init__(self, df, seq_len=52, pred_len=8, min_target_date=None):
        self.samples = []
        skipped = 0

        for (store, dept), grp in df.groupby(['Store', 'Dept']):
            grp = grp.sort_values('Date').reset_index(drop=True)
            sales = grp['Weekly_Sales'].values.astype(np.float32)
            dates = grp['Date'].values
            holidays = grp['IsHoliday'].values.astype(np.float32)
            weights = grp['holiday_weight'].values.astype(np.float32)

            if len(sales) < seq_len + pred_len:
                skipped += 1
                continue

            mu = sales.mean()
            std = sales.std() + 1e-8
            sales_norm = (sales - mu) / std

            for i in range(len(sales) - seq_len - pred_len + 1):
                target_date = dates[i + seq_len]
                if min_target_date is not None and target_date < min_target_date:
                    continue
                self.samples.append({
                    'x': sales_norm[i:i + seq_len],
                    'y': sales[i + seq_len:i + seq_len + pred_len],
                    'y_norm': sales_norm[i + seq_len:i + seq_len + pred_len],
                    'holiday': holidays[i + seq_len:i + seq_len + pred_len],
                    'weight': weights[i + seq_len:i + seq_len + pred_len],
                    'mu': mu,
                    'std': std,
                })

        log_progress(f'Dataset windows: {len(self.samples):,} | skipped short series: {skipped}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return (
            torch.from_numpy(s['x']),
            torch.from_numpy(s['y_norm']),
            torch.from_numpy(s['y']),
            torch.from_numpy(s['holiday']),
            torch.from_numpy(s['weight']),
            torch.tensor(s['mu'], dtype=torch.float32),
            torch.tensor(s['std'], dtype=torch.float32),
        )


@torch.no_grad()
def evaluate_wmae(model, loader):
    model.eval()
    num, denom = 0.0, 0.0
    for x, _, y_raw, _, w, mu, std in loader:
        x = x.to(device)
        pred_norm = model(x).detach().cpu().numpy()
        pred_norm = np.nan_to_num(pred_norm, nan=0.0, posinf=0.0, neginf=0.0)
        y_raw_np = y_raw.numpy().astype(np.float64)
        w_np = np.nan_to_num(w.numpy().astype(np.float64), nan=1.0)
        mu_np = mu.numpy().reshape(-1, 1)
        std_np = std.numpy().reshape(-1, 1)
        pred_raw = np.clip(pred_norm * std_np + mu_np, 0, None)
        num += np.sum(w_np * np.abs(y_raw_np - pred_raw))
        denom += np.sum(w_np)
    if denom <= 0:
        return float('inf')
    score = num / denom
    return float(score) if np.isfinite(score) else float('inf')


@torch.no_grad()
def evaluate_val_loss(model, loader, criterion):
    model.eval()
    total, n = 0.0, 0
    for x, y_norm, *_ in loader:
        x = x.to(device)
        y_norm = y_norm.to(device)
        pred = model(x)
        total += criterion(pred, y_norm).item()
        n += 1
    return total / max(n, 1)


def train_model(model, train_loader, val_loader, config, run_name):
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=config.get('lr_patience', 10), min_lr=config['lr'] * 1e-2
    )
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_wmae = float('inf')
    best_state = None
    patience = 0

    log_progress(f'Start training: {run_name} | params={count_params(model):,}')

    for epoch in range(1, config['epochs'] + 1):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for x, y_norm, _, _, _, _, _ in train_loader:
            x = x.to(device)
            y_norm = y_norm.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y_norm)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        val_loss = evaluate_val_loss(model, val_loader, criterion)
        val_wmae = evaluate_wmae(model, val_loader)
        scheduler.step(val_loss)

        epoch_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if best_state is None or val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = epoch_state
            if np.isfinite(val_wmae):
                best_wmae = val_wmae
            patience = 0
            marker = ' *best*'
        else:
            patience += 1
            marker = ''

        ckpt_path = CHECKPOINT_DIR / f'{run_name}_epoch{epoch:03d}.pt'
        torch.save({'epoch': epoch, 'model_state': epoch_state, 'val_loss': val_loss, 'val_wmae': val_wmae, 'config': config}, ckpt_path)

        wmae_str = f'{val_wmae:,.2f}' if np.isfinite(val_wmae) else 'inf'
        log_progress(
            f'{run_name} epoch {epoch}/{config["epochs"]} loss={total_loss/max(n_batches,1):.4f} '
            f'val_loss={val_loss:.4f} val_wmae={wmae_str}{marker}'
        )

        if patience >= config['patience']:
            log_progress(f'{run_name} early stop at epoch {epoch}')
            break

    if best_state is None:
        best_state = epoch_state

    model.load_state_dict(best_state)
    best_ckpt = CHECKPOINT_DIR / f'{run_name}_best.pt'
    torch.save({'model_state': best_state, 'config': config, 'best_val_loss': best_val_loss, 'best_val_wmae': best_wmae}, best_ckpt)

    final_wmae = evaluate_wmae(model, val_loader)
    log_progress(f'{run_name} finished | best val loss={best_val_loss:.4f} | val WMAE={final_wmae:,.2f}')
    return final_wmae, best_val_loss, model

In [ ]:
#7
log_progress('Building datasets')

model_train_df = train_df
model_val_df = val_df
if MAX_SERIES is not None:
    top_pairs = (
        train_df.groupby(['Store', 'Dept']).size()
        .sort_values(ascending=False)
        .head(MAX_SERIES)
        .index.tolist()
    )
    pair_set = set(top_pairs)
    model_train_df = train_df[[(s, d) in pair_set for s, d in zip(train_df['Store'], train_df['Dept'])]].copy()
    model_val_df = val_df[[(s, d) in pair_set for s, d in zip(val_df['Store'], val_df['Dept'])]].copy()
    log_progress(f'Using top {MAX_SERIES} store-dept series')

train_ds = WalmartDLinearDataset(model_train_df, seq_len=SEQ_LEN, pred_len=PRED_LEN)
full_for_val = pd.concat([model_train_df, model_val_df]).sort_values(['Store', 'Dept', 'Date'])
val_ds = WalmartDLinearDataset(full_for_val, seq_len=SEQ_LEN, pred_len=PRED_LEN, min_target_date=SPLIT_DATE)

assert len(train_ds) > 0 and len(val_ds) > 0, 'Empty train/val dataset'

BATCH_SIZE = 1024 if USE_CUDA else 512
PIN_MEMORY = USE_CUDA
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)

log_progress(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | batch_size={BATCH_SIZE}')

In [ ]:
#8
CONFIG_BASELINE = {
    'lookback': SEQ_LEN,
    'horizon': PRED_LEN,
    'kernel_size': 25,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'epochs': 150 if LONG_RUN else 80,
    'patience': 50 if LONG_RUN else 25,
    'lr_patience': 10,
    'batch_size': BATCH_SIZE,
    'device': str(device),
    'long_run': LONG_RUN,
    'note': 'DLinear baseline — kernel_size=25',
}

model_baseline = DLinear(SEQ_LEN, PRED_LEN, kernel_size=CONFIG_BASELINE['kernel_size']).to(device)

baseline_wmae, baseline_val_loss, model_baseline = train_model(
    model_baseline, train_loader, val_loader, CONFIG_BASELINE, 'DLinear_Baseline'
)

with mlflow.start_run(run_name='DLinear_Baseline'):
    mlflow.log_params(CONFIG_BASELINE)
    mlflow.log_metric('val_wmae', baseline_wmae)
    mlflow.log_metric('val_loss', baseline_val_loss)
    mlflow.log_metric('n_params', count_params(model_baseline))
    mlflow.log_artifact(str(CHECKPOINT_DIR / 'DLinear_Baseline_best.pt'), 'checkpoints')

In [ ]:
#9
CONFIG_MEDIUM = {
    **CONFIG_BASELINE,
    'kernel_size': 51,
    'lr': 5e-4,
    'weight_decay': 1e-3,
    'epochs': 150 if LONG_RUN else 100,
    'patience': 50 if LONG_RUN else 30,
    'note': 'DLinear medium — longer seasonal kernel (51)',
}

model_medium = DLinear(SEQ_LEN, PRED_LEN, kernel_size=CONFIG_MEDIUM['kernel_size']).to(device)

medium_wmae, medium_val_loss, model_medium = train_model(
    model_medium, train_loader, val_loader, CONFIG_MEDIUM, 'DLinear_Medium'
)

with mlflow.start_run(run_name='DLinear_Medium'):
    mlflow.log_params(CONFIG_MEDIUM)
    mlflow.log_metric('val_wmae', medium_wmae)
    mlflow.log_metric('val_loss', medium_val_loss)
    mlflow.log_metric('n_params', count_params(model_medium))
    mlflow.log_metric('improvement_over_baseline', baseline_wmae - medium_wmae)
    mlflow.log_artifact(str(CHECKPOINT_DIR / 'DLinear_Medium_best.pt'), 'checkpoints')

In [ ]:
#10
if np.isfinite(baseline_wmae) and np.isfinite(medium_wmae):
    pick_medium = medium_wmae <= baseline_wmae
elif np.isfinite(medium_wmae):
    pick_medium = True
elif np.isfinite(baseline_wmae):
    pick_medium = False
else:
    pick_medium = medium_val_loss <= baseline_val_loss

if pick_medium:
    best_name = 'DLinear_Medium'
    best_wmae = medium_wmae
    best_val_loss = medium_val_loss
    best_model = model_medium
    best_config = CONFIG_MEDIUM
else:
    best_name = 'DLinear_Baseline'
    best_wmae = baseline_wmae
    best_val_loss = baseline_val_loss
    best_model = model_baseline
    best_config = CONFIG_BASELINE

results = {'DLinear_Baseline': baseline_wmae, 'DLinear_Medium': medium_wmae}
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
names = list(results.keys())
scores = [v if np.isfinite(v) else 0 for v in results.values()]
colors = ['#8b949e', '#f78166' if best_name == 'DLinear_Medium' else '#58a6ff']
if best_name == 'DLinear_Baseline':
    colors = ['#f78166', '#58a6ff']
bars = ax.bar(names, scores, color=colors, width=0.45)
label_vals = list(results.values())
ax.bar_label(bars, labels=[f'{v:,.1f}' if np.isfinite(v) else 'inf' for v in label_vals], color='white')
ax.set_title('DLinear runs (val WMAE, lower is better)', color='white')
ax.set_ylabel('WMAE', color='#8b949e')
ax.tick_params(colors='#8b949e')
plt.tight_layout()
plt.savefig('dlinear_all_runs.png', dpi=130, facecolor='#0f1117')
plt.show()

torch.save({
    'model_state_dict': best_model.state_dict(),
    'config': best_config,
    'best_run': best_name,
    'best_val_wmae': best_wmae,
    'seq_len': SEQ_LEN,
    'pred_len': PRED_LEN,
}, WORK_DIR / 'dlinear_best.pt')

with open(WORK_DIR / 'dlinear_best_pipeline_spec.json', 'w', encoding='utf-8') as f:
    json.dump({
        'preprocess': {'sort_by': ['Store', 'Dept', 'Date'], 'target_scaling': 'per_series_mean_std'},
        'model': {'architecture': 'DLinear', **best_config},
        'selection': {'best_run': best_name, 'criterion': 'val_wmae'},
    }, f, indent=2)

with mlflow.start_run(run_name='DLinear_Best'):
    mlflow.log_param('best_run', best_name)
    mlflow.log_params(best_config)
    mlflow.log_metric('best_val_wmae', best_wmae)
    mlflow.log_metric('baseline_val_wmae', baseline_wmae)
    mlflow.log_metric('medium_val_wmae', medium_wmae)
    mlflow.log_metric('best_val_loss', best_val_loss)
    mlflow.log_artifact(str(WORK_DIR / 'dlinear_best.pt'), 'model')
    mlflow.log_artifact(str(WORK_DIR / 'dlinear_best_pipeline_spec.json'), 'pipeline')
    mlflow.log_artifact('dlinear_all_runs.png', 'plots')
    best_model_cpu = best_model.to('cpu').eval()
    try:
        mlflow.pytorch.log_model(
            best_model_cpu,
            artifact_path='dlinear-best-model',
            registered_model_name='DLinear_Walmart_Best',
            serialization_format='pickle',
        )
        log_progress('Registered MLflow model: DLinear_Walmart_Best')
    except Exception as e:
        log_progress(f'MLflow registration skipped: {e}')

log_progress(f'Best model: {best_name} | WMAE={best_wmae:,.2f} | val_loss={best_val_loss:.4f}')

In [ ]:
#11
summary = {
    'status': 'COMPLETE',
    'finished_at_utc': datetime.utcnow().isoformat() + 'Z',
    'device': str(device),
    'notebook_version': NOTEBOOK_VERSION,
    'long_run': LONG_RUN,
    'baseline_wmae': float(baseline_wmae),
    'medium_wmae': float(medium_wmae),
    'best_run': best_name,
    'best_wmae': float(best_wmae),
    'artifacts': [
        str(WORK_DIR / 'dlinear_best.pt'),
        str(PROGRESS_LOG),
        str(WORK_DIR / 'dlinear_all_runs.png'),
    ],
}

with open(RUN_COMPLETE, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

log_progress('RUN COMPLETE — open RUN_COMPLETE.txt on Kaggle to verify success')
print('\n=== NOTEBOOK FINISHED SUCCESSFULLY ===')
print(json.dumps(summary, indent=2))

In [ ]:
#12 — W&B backfill (optional; never fails the notebook)
import json, re
import wandb
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
LOG = WORK_DIR / 'dlinear_progress.log'

_wandb_ok = False
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
    if not _wandb_key or len(_wandb_key) < 40:
        raise RuntimeError('WANDB_API_KEY missing or too short (use full secret, not Key ID)')
    wandb.login(key=_wandb_key)
    _wandb_ok = True
except Exception as e:
    print(f'W&B skipped: {e}')

if not _wandb_ok:
    print('Training still OK — check RUN_COMPLETE.txt and MLflow')
elif not LOG.exists():
    print('W&B skipped: dlinear_progress.log not found')
elif not (WORK_DIR / 'RUN_COMPLETE.txt').exists():
    print('W&B skipped: RUN_COMPLETE.txt missing (re-run after training finishes)')
else:
    EPOCH_RE = re.compile(
        r'(?P<run>DLinear_\w+) epoch (?P<epoch>\d+)/\d+ '
        r'loss=(?P<train_loss>[\d.]+) val_loss=(?P<val_loss>[\d.]+) val_wmae=(?P<val_wmae>[\d,.]+|inf)'
    )

    epochs_by_run = {}
    for line in LOG.read_text().splitlines():
        m = EPOCH_RE.search(line)
        if not m:
            continue
        run_name = m.group('run')
        wmae_raw = m.group('val_wmae').replace(',', '')
        val_wmae = float('inf') if wmae_raw == 'inf' else float(wmae_raw)
        epochs_by_run.setdefault(run_name, []).append({
            'epoch': int(m.group('epoch')),
            'train_loss': float(m.group('train_loss')),
            'val_loss': float(m.group('val_loss')),
            'val_wmae': val_wmae,
        })

    for run_name, rows in epochs_by_run.items():
        by_epoch = {r['epoch']: r for r in rows}
        rows = [by_epoch[e] for e in sorted(by_epoch)]
        wb = wandb.init(project='walmart-forecasting', name=run_name, reinit=True)
        for row in rows:
            wandb.log(row)
        wb.finish()
        print(f'Logged {len(rows)} epochs -> {run_name}')

    summary = json.loads((WORK_DIR / 'RUN_COMPLETE.txt').read_text())
    wb = wandb.init(project='walmart-forecasting', name='DLinear_Summary', reinit=True)
    wandb.log({
        'baseline_wmae': summary['baseline_wmae'],
        'medium_wmae': summary['medium_wmae'],
        'best_wmae': summary['best_wmae'],
    })
    if (WORK_DIR / 'dlinear_all_runs.png').exists():
        wandb.log({'chart': wandb.Image(str(WORK_DIR / 'dlinear_all_runs.png'))})
    wb.finish()
    print('Done: https://wandb.ai — walmart-forecasting')